# json2vec Hello World

This notebook trains a tiny model from an in-memory synthetic dataset. No files, S3 bucket, logger, or checkpoint setup is required.

In [ ]:
import lightning.pytorch as lit
import torch

from json2vec import Dataset, DefaultDataModule, Hyperparameters, JSON2Vec, Strata, Suffix, shim
from json2vec.data.datasets import encode

BATCH_SIZE = 4

In [ ]:
@shim(yields=True)
def hello_world_records(_observation: dict, strata: Strata):
    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
        {"color": "red", "label": "warm"},
        {"color": "blue", "label": "cool"},
    ]

    if strata == Strata.predict:
        for record in records[:2]:
            yield {"color": record["color"]}
        return

    yield from records

In [3]:
params = Hyperparameters.model_validate(
    {
        "d_model": 16,
        "target": ["record/label"],
        "fields": {
            "name": "record",
            "type": "array",
            "attention": "none",
            "max_length": 1,
            "n_outputs": 1,
            "fields": [
                {"name": "color", "type": "category", "query": "[*].color", "max_vocab_size": 16},
                {"name": "label", "type": "category", "query": "[*].label", "max_vocab_size": 8, "topk": [2]},
            ],
        },
    }
)

data = Dataset(
    root=None,
    sample_rate=1.0,
    file_buffer_size=1,
    observation_buffer_size=32,
    processor="hello_world_records",
    suffix=Suffix.ndjson,
    patterns={strata: r".*" for strata in Strata},
)

model = JSON2Vec(
    hyperparameters=params,
    batch_size=BATCH_SIZE,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = DefaultDataModule.from_model(
    model,
    dataset=data,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
)

2026-05-17 17:08:00.396 | INFO     | json2vec.architecture.root:__init__:127 - initialized JSON2Vec module


In [ ]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=5,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
    enable_progress_bar=False,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-05-17 17:08:00.427 | INFO     | json2vec.data.datasets:dataloader:586 - building dataloader
2026-05-17 17:08:00.427 | INFO     | json2vec.data.datasets:__init__:539 - initialized batch dataset
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-pac

In [10]:
inputs = encode(
    batch=[[{"color": "red"}], [{"color": "blue"}]],
    hyperparameters=model.hyperparameters,
    strata=Strata.predict,
    state=model.state,
)

model.eval()
with torch.no_grad():
    predictions = model(inputs)

output, embeddings = model.write(predictions)

output['record/label']

{'state': {'valued': [[0.8342441916465759], [0.8997030854225159]],
  'null': [[0.043644070625305176], [0.02467968501150608]],
  'padded': [[0.03431139513850212], [0.030663592740893364]],
  'masked': [[0.04322353005409241], [0.0212665107101202]],
  'other': [[0.044576793909072876], [0.02368716523051262]]},
 'content': {'value': [['warm'], ['cool']],
  'probability': [[0.8508769273757935], [0.7458580136299133]],
  'topk': [[[{'label': 'warm', 'probability': 0.8508769273757935},
     {'label': 'cool', 'probability': 0.14912311732769012}]],
   [[{'label': 'cool', 'probability': 0.7458580136299133},
     {'label': 'warm', 'probability': 0.2541419565677643}]]]}}